# Phase 3: local models, GPU accuracy run B (llama3.2:3b, phi3.5:3.8b, gemma3:4b) + optional GPU-server tier

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only**.

Same settings as notebook A. Section 1 runs the three mid-size core models (240 dev tasks each). **Section 2 is OPTIONAL**: the 'own GPU server' tier (gemma3:12b, phi4:14b, qwen2.5:14b), reported separately, not part of the 16 models. Run it only if the session time allows (~1-2 h; models of 7-9 GB are pulled and deleted one at a time). Both write to the same JSONL and are resumable.

At the end download **`phase3_local_gpu_b.zip`** into `eval/results/phase3/`.

In [ ]:
%cd /kaggle/working
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo no GPU
!nproc
!df -h /kaggle/working | tail -1

In [ ]:
%%writefile phase3_run.py
"""Phase 3 local runner: Ollama on Kaggle, dev split, per-category system prompts, temperature 0.
Output: JSONL, one answer per line (fields as in eval/answers.py + done_reason, eval_ms, load_ms). Resumable.
"""
import argparse, json, os, shutil, subprocess, sys, time, urllib.request

HOST = "http://127.0.0.1:11434"
SYSTEM = json.load(open("system_prompts.json", encoding="utf-8"))
NUM_PREDICT = {"math": 768, "logic": 768, "sentiment": 256, "ner": 384, "code_debug": 1024,
               "code_gen": 1024, "summarisation": 256, "factual": 320}


def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)


def install_and_serve():
    if not shutil.which("ollama"):
        sh("apt-get update -qq && apt-get install -y -qq zstd >/dev/null 2>&1 || true")
        sh("curl -fsSL https://ollama.com/install.sh | sh")
    try:
        urllib.request.urlopen(HOST + "/api/version", timeout=2)
        return
    except Exception:
        pass
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(HOST + "/api/version", timeout=2)
            return
        except Exception:
            time.sleep(1)
    sys.exit("ollama server did not start; see /tmp/ollama.log")


def chat(model, category, prompt, cpu):
    opts = {"temperature": 0, "num_predict": NUM_PREDICT[category], "num_ctx": 4096}
    if cpu:
        opts.update(num_gpu=0, num_thread=2)
    payload = {"model": model, "stream": False, "keep_alive": "10m",
               "messages": [{"role": "system", "content": SYSTEM[category]},
                            {"role": "user", "content": prompt}],
               "options": opts}
    if model.startswith("qwen3"):
        payload["think"] = False
    req = urllib.request.Request(HOST + "/api/chat", json.dumps(payload).encode(), {"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=1800) as r:
        d = json.load(r)
    return d, int((time.perf_counter() - t0) * 1000)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", default="tasks_dev.json")
    ap.add_argument("--ids", default=None, help="JSON file with {'ids': [...]} to restrict the tasks")
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--run-id", required=True)
    ap.add_argument("--cpu", action="store_true", help="num_gpu=0, num_thread=2")
    a = ap.parse_args()
    tasks = json.load(open(a.tasks, encoding="utf-8"))
    if a.ids:
        keep = set(json.load(open(a.ids))["ids"])
        tasks = [t for t in tasks if t["id"] in keep]
    install_and_serve()
    done = set()
    if os.path.exists(a.out):
        for line in open(a.out, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                done.add((r["model"], r["task_id"]))
    for model in a.models:
        todo = [t for t in tasks if (model, t["id"]) not in done]
        if not todo:
            print(model, "already complete")
            continue
        sh("ollama pull " + model)
        try:  # warm-up so the first recorded answer does not include model load time (not saved)
            chat(model, "sentiment", "The food was fine.", a.cpu)
        except Exception as e:
            print("warm-up failed", model, e)
        n = 0
        for t in todo:
            rec = {"task_id": t["id"], "model": model, "provider": "ollama-kaggle", "run_id": a.run_id,
                   "raw_text": "", "extracted": None, "tokens_in": None, "tokens_out": None,
                   "tokens_reasoning": None, "latency_ms": None, "cost_usd": 0.0, "error": None,
                   "done_reason": None, "eval_ms": None, "load_ms": None}
            try:
                d, ms = chat(model, t["category"], t["prompt"], a.cpu)
                msg = d.get("message", {})
                rec.update(raw_text=msg.get("content", ""), tokens_in=d.get("prompt_eval_count"),
                           tokens_out=d.get("eval_count"), latency_ms=ms, done_reason=d.get("done_reason"),
                           eval_ms=(d.get("eval_duration") or 0) // 1_000_000,
                           load_ms=(d.get("load_duration") or 0) // 1_000_000)
                if msg.get("thinking"):
                    rec["raw_text"] = "<think>" + msg["thinking"] + "</think>" + rec["raw_text"]
            except Exception as e:
                rec["error"] = str(e)[:300]
            with open(a.out, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
            if n % 20 == 0:
                print(model, n, "answered", flush=True)
        sh("ollama stop " + model + " || true")
        sh("ollama rm " + model)
    print("done ->", a.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile system_prompts.json
{
 "math": "Solve the problem. You may show brief working. End with a final line exactly of the form `ANSWER: <number>` (digits only, no units, no commas).",
 "logic": "Solve the puzzle. You may show brief working. End with a final line exactly of the form `ANSWER: <value>`, where <value> is only the name/item/position asked for.",
 "sentiment": "Classify the sentiment as exactly one of: positive, negative, mixed, neutral. Give a one-sentence reason, then end with a final line exactly of the form `ANSWER: <label>`.",
 "ner": "Extract the named entities. Return only JSON: a list of objects {\"text\": \"<exact span>\", \"type\": \"PERSON|ORG|LOCATION|DATE\"}. No other text.",
 "code_debug": "Find the bug and return only the corrected function, complete, in a single Python code block. No explanation outside the code block.",
 "code_gen": "Return only the requested function in a single Python code block, with the exact name and signature given. No explanation outside the code block.",
 "summarisation": "Follow the length/format constraint in the request exactly. Use only facts stated in the text; do not add anything. Return only the summary.",
 "factual": "Answer accurately and concisely, in at most 120 words. Do not pad."
}

In [ ]:
%%writefile tasks_dev.json
[
 {
  "id": "fact-002",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "quick question - when and how did the Western Roman Empire end?"
 },
 {
  "id": "fact-003",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What does GDP measure?"
 },
 {
  "id": "fact-007",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "What caused the Dust Bowl in the 1930s United States?"
 },
 {
  "id": "fact-009",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "What is the difference between a recession and a depression?"
 },
 {
  "id": "fact-010",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "Why does the Moon have phases?"
 },
 {
  "id": "fact-012",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is the difference between a virus and a bacterium?"
 },
 {
  "id": "fact-013",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "quick question - why is the speed of light central to special relativity?"
 },
 {
  "id": "fact-015",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What was the Industrial Revolution?"
 },
 {
  "id": "fact-017",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "I'm writing a short explainer and I want to be sure I get this right. What was the significance of the Magna Carta of 1215?"
 },
 {
  "id": "fact-018",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is a CPU cache and why does it exist?"
 },
 {
  "id": "fact-019",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "Why do central banks raise interest rates to fight inflation, and what is the main trade-off?"
 },
 {
  "id": "fact-021",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is the greenhouse effect?"
 },
 {
  "id": "fact-022",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "How does a heat pump work, and why can it deliver more heat than the electrical energy it uses?"
 },
 {
  "id": "fact-023",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "quick question - why is the sky blue?"
 },
 {
  "id": "fact-025",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is a bond in finance?"
 },
 {
  "id": "fact-026",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "I'm writing a short explainer and I want to be sure I get this right. How do vaccines work?"
 },
 {
  "id": "fact-028",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "Why is the sky blue and not violet?"
 },
 {
  "id": "fact-030",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "What is a race condition in programming?"
 },
 {
  "id": "fact-032",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "quick question - how does the attention mechanism in a transformer model work?"
 },
 {
  "id": "fact-033",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "Explain what an API is."
 },
 {
  "id": "fact-034",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "Does the Coriolis effect decide which way water spins down a bathtub drain?"
 },
 {
  "id": "fact-036",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "Why do we have leap years, and why aren't all years divisible by 4 leap years?"
 },
 {
  "id": "fact-037",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is machine learning?"
 },
 {
  "id": "fact-038",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is inflation?"
 },
 {
  "id": "fact-039",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What was the Cold War?"
 },
 {
  "id": "fact-040",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "quick question - what does the Pythagorean theorem state?"
 },
 {
  "id": "fact-041",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "I'm writing a short explainer and I want to be sure I get this right. How does public-key cryptography let two strangers communicate securely?"
 },
 {
  "id": "fact-043",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "What is the difference between supervised and unsupervised learning?"
 },
 {
  "id": "fact-044",
  "category": "factual",
  "difficulty": "easy",
  "prompt": "What is compound interest?"
 },
 {
  "id": "fact-048",
  "category": "factual",
  "difficulty": "hard",
  "prompt": "What does the Heisenberg uncertainty principle say?"
 },
 {
  "id": "math-003",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Quick one for the inventory report - we opened the quarter with 1940 bottles of juice. Week 1 we lost 40% of that to sales, week 2 we sold 50% of whatever was left, and in week 3 a further 45 went out. What's the closing stock?"
 },
 {
  "id": "math-004",
  "category": "math",
  "difficulty": "hard",
  "prompt": "₹8,500 is invested at 20% compound interest per year. After two years, an extra ₹1,500 is added, and the whole balance then grows for one more year at the same rate. What is the final balance?"
 },
 {
  "id": "math-005",
  "category": "math",
  "difficulty": "easy",
  "prompt": "Work out how many bags of rice remain. Starting stock: 1900. Monday sales: 25% of the stock. Tuesday sales: 10 units."
 },
 {
  "id": "math-007",
  "category": "math",
  "difficulty": "hard",
  "prompt": "A prize of ₹672 is shared among three people in the ratio 3:4:5. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-008",
  "category": "math",
  "difficulty": "hard",
  "prompt": "A prize of ₹336 is shared among three people in the ratio 1:2:4. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-010",
  "category": "math",
  "difficulty": "easy",
  "prompt": "drove 180 km doing 45 km/h, then 180 km at 30 km/h. total time on the road in hours?"
 },
 {
  "id": "math-011",
  "category": "math",
  "difficulty": "easy",
  "prompt": "A recipe that serves 4 people uses 300 g of flour. How many grams of flour are needed to serve 10 people with the same recipe?"
 },
 {
  "id": "math-014",
  "category": "math",
  "difficulty": "easy",
  "prompt": "our village has 8,800 residents and it keeps growing by 25% a year. what will the population be 2 years from now?"
 },
 {
  "id": "math-015",
  "category": "math",
  "difficulty": "easy",
  "prompt": "A shop has 1680 chairs. It sells 50% of them on Monday and 40 more on Tuesday. How many chairs are left?"
 },
 {
  "id": "math-017",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Quick one for the inventory report - we opened the quarter with 900 tickets. Week 1 we lost 40% of that to sales, week 2 we sold 20% of whatever was left, and in week 3 a further 25 went out. What's the closing stock?"
 },
 {
  "id": "math-018",
  "category": "math",
  "difficulty": "hard",
  "prompt": "For the board deck: we began with 11,400 subscribers; year 1 grew 25%, year 2 shrank 20%, year 3 grew 10%. What is the year-3 subscriber count?"
 },
 {
  "id": "math-019",
  "category": "math",
  "difficulty": "hard",
  "prompt": "A prize of ₹100 is shared among three people in the ratio 1:3:6. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-022",
  "category": "math",
  "difficulty": "easy",
  "prompt": "bought a lamp wholesale at ₹1400, tagged it 40% higher. how many rupees of markup is that?"
 },
 {
  "id": "math-024",
  "category": "math",
  "difficulty": "easy",
  "prompt": "Split 300 rupees in the ratio 2:3. Report the bigger share."
 },
 {
  "id": "math-025",
  "category": "math",
  "difficulty": "easy",
  "prompt": "The price of a bike is raised by 20% and then the new price is cut by 20%. If the bike started at ₹500, what does it cost at the end?"
 },
 {
  "id": "math-026",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Shift summary needed: P ran for the full 3 hours at 12 parts/hour; Q joined for just the final 2 hours at 10 parts/hour. Total parts from both?"
 },
 {
  "id": "math-028",
  "category": "math",
  "difficulty": "easy",
  "prompt": "i handed over ₹500 for 7 notebooks that cost ₹60 apiece - what should i get back?"
 },
 {
  "id": "math-030",
  "category": "math",
  "difficulty": "hard",
  "prompt": "A company has 7,000 subscribers. Over three consecutive years its subscriber count changes by +25%, -50% and +20% respectively. How many subscribers does it have at the end of year three?"
 },
 {
  "id": "math-031",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Planning my savings: ₹9,500 sits in an account earning 20% a year, compounded annually. At the end of year 2 I top it up with ₹1,000, and leave everything in for year 3. What's the balance after year 3?"
 },
 {
  "id": "math-032",
  "category": "math",
  "difficulty": "hard",
  "prompt": "A customer purchases 6 units of item P at ₹45 and 2 units of item Q at ₹125. A 25% discount applies to the total. If the customer tenders ₹2000, compute the change returned."
 },
 {
  "id": "math-036",
  "category": "math",
  "difficulty": "easy",
  "prompt": "so Vikram's store started the week with 680 oranges, sold 25% on Monday, then another 60 on Tuesday. how many are still on the shelf?"
 },
 {
  "id": "math-037",
  "category": "math",
  "difficulty": "easy",
  "prompt": "me and my cousin are splitting ₹480 in the ratio 3 to 5 (he gets the bigger part). how much is his share?"
 },
 {
  "id": "math-038",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Shift summary needed: P ran for the full 2 hours at 15 parts/hour; Q joined for just the final 1 hours at 16 parts/hour. Total parts from both?"
 },
 {
  "id": "math-041",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Divya is 4 times as old as Divya's daughter. In 17 years Divya will be 3 times as old as the daughter will be then. How old is the daughter now?"
 },
 {
  "id": "math-043",
  "category": "math",
  "difficulty": "easy",
  "prompt": "i handed over ₹1000 for 3 notebooks that cost ₹18 apiece - what should i get back?"
 },
 {
  "id": "math-044",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Section A has 30 students with an average mark of 59. Section B has 30 students with an average mark of 65. What is the average mark of all 60 students together?"
 },
 {
  "id": "math-045",
  "category": "math",
  "difficulty": "easy",
  "prompt": "our press stamps out 336 bolts every 12 min. running it nonstop for 4 hours, how many bolts is that?"
 },
 {
  "id": "math-046",
  "category": "math",
  "difficulty": "easy",
  "prompt": "A town has 6,800 people and its population grows 20% every year. How many people live there after 2 years?"
 },
 {
  "id": "math-049",
  "category": "math",
  "difficulty": "easy",
  "prompt": "our press stamps out 136 bolts every 8 min. running it nonstop for 2 hours, how many bolts is that?"
 },
 {
  "id": "math-050",
  "category": "math",
  "difficulty": "hard",
  "prompt": "Rohan buys 4 pens at ₹45 each and 5 folders at ₹80 each. The whole bill gets a 25% discount. Rohan pays with a ₹2000 note. How much change comes back?"
 },
 {
  "id": "sent-001",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nI'm furious - I've been charged twice and nobody will help me.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-002",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Classify the sentiment of this review as positive, negative, mixed or neutral, then give a one-line reason: 'The laptop has 16 GB of RAM and a 512 GB SSD.'"
 },
 {
  "id": "sent-003",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"Food was cold, the waiter was rude, and we waited an hour. Never again.\""
 },
 {
  "id": "sent-004",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"The film depicts the devastating famine of 1943 and is based on historical records.\""
 },
 {
  "id": "sent-006",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"Not bad at all - honestly the best ramen I've had outside Japan.\""
 },
 {
  "id": "sent-009",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nSupport was friendly and apologetic, yet the issue is still not fixed after three weeks.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-011",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"Lovely screen and sound, but the battery barely lasts a day.\""
 },
 {
  "id": "sent-012",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nFour stars for the product, one star for the courier who crushed the box.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-013",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"Decent camera and a gorgeous display; the speaker is tinny and the software is buggy.\""
 },
 {
  "id": "sent-015",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Classify the sentiment of this tweet as positive, negative, mixed or neutral, then give a one-line reason: 'ngl this album slaps, been on repeat all week'"
 },
 {
  "id": "sent-017",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Classify the sentiment of this message as positive, negative, mixed or neutral, then give a one-line reason: 'Our meeting is scheduled for Thursday at 3 pm in room 204.'"
 },
 {
  "id": "sent-019",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this message - positive, negative, mixed, or neutral? \"The store opens at 9 am and closes at 6 pm on weekdays.\""
 },
 {
  "id": "sent-022",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"The hotel room was spotless, the staff were lovely and the breakfast was fantastic.\""
 },
 {
  "id": "sent-023",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nI didn't expect much from a budget phone, yet here I am, a month in, with zero complaints.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-026",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"Great value for money; these headphones sound far better than I expected.\""
 },
 {
  "id": "sent-027",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Classify the sentiment of this review as positive, negative, mixed or neutral, then give a one-line reason: 'The battery died after three days and support never replied. Total waste of money.'"
 },
 {
  "id": "sent-028",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Is this review positive, negative, mixed or neutral? Explain briefly.\n\n\"Worst movie I've seen all year; the plot made no sense and I nearly fell asleep.\""
 },
 {
  "id": "sent-029",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Classify the sentiment of this message as positive, negative, mixed or neutral, then give a one-line reason: 'Thanks for nothing - three calls, two transfers, and I'm still waiting on the refund.'"
 },
 {
  "id": "sent-032",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "what's the vibe of this tweet - positive, negative, mixed, or neutral? \"Wow, a whole hour to deliver a cold sandwich. Truly a new record.\""
 },
 {
  "id": "sent-033",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"The app update is wonderful. Everything loads instantly now.\""
 },
 {
  "id": "sent-034",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"The course content is excellent; the platform itself is clunky and keeps logging me out.\""
 },
 {
  "id": "sent-035",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"The cafe was fine. Nothing special, nothing wrong.\""
 },
 {
  "id": "sent-036",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Is this message positive, negative, mixed or neutral? Explain briefly.\n\n\"If I could give the repair crew six stars I would; they stayed late, fixed it properly and cleaned up after themselves.\""
 },
 {
  "id": "sent-037",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Classify the sentiment of this review as positive, negative, mixed or neutral, then give a one-line reason: 'Great location and friendly host, but the room was noisy and small.'"
 },
 {
  "id": "sent-039",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "Classify the sentiment of this tweet as positive, negative, mixed or neutral, then give a one-line reason: 'This update ruined the app. It crashes every time I open it.'"
 },
 {
  "id": "sent-044",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "what's the vibe of this review - positive, negative, mixed, or neutral? \"The room was 'sea view' if you stood on the toilet and squinted.\""
 },
 {
  "id": "sent-045",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nIt's an average phone. Does what phones do.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-046",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "Classify the sentiment of this review as positive, negative, mixed or neutral, then give a one-line reason: 'Honestly the food is the best in town. The staff, though, act like you're bothering them.'"
 },
 {
  "id": "sent-048",
  "category": "sentiment",
  "difficulty": "hard",
  "prompt": "We're triaging customer feedback. Here is one entry:\n\nNot thrilled with the price, but I can't deny the quality is excellent.\n\nTag its overall sentiment (positive / negative / mixed / neutral) and say why in a sentence."
 },
 {
  "id": "sent-049",
  "category": "sentiment",
  "difficulty": "easy",
  "prompt": "what's the vibe of this message - positive, negative, mixed, or neutral? \"The report has 14 pages and includes three appendices.\""
 },
 {
  "id": "summ-004",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text as exactly 3 bullet points (each line starting with '- '):\n\nWildlife officials counted 3,200 snow leopards across Central Asia in a five-year survey, up from an estimate of 2,500 a decade ago. Conservationists credit community programmes that pay herders compensation for livestock lost to leopards, which has reduced revenge killings by about half in surveyed valleys. However, the survey used camera traps in only 40 percent of the range, and some experts believe the apparent rise may partly reflect better counting methods rather than a real increase. Climate change is shrinking the high-altitude habitat the animals depend on. Snow leopards live in 12 countries at altitudes up to 5,500 metres and are very hard to spot, which is why researchers rely on camera traps and droppings. They are listed as vulnerable. The survey was coordinated by a conservation coalition and took five years, covering about 1.2 million square kilometres."
 },
 {
  "id": "summ-007",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Condense this passage in exactly one sentence. Keep only facts stated in the text.\n\n\"A software update released on Tuesday fixes a security flaw that could let attackers read messages sent through the Chatly app. The flaw, found by an independent researcher, affected versions released since September and was present on both phones and computers. Chatly says it has seen no evidence that the weakness was used. Users should install version 5.3 as soon as possible; the update downloads automatically if automatic updates are switched on. Chatly has 40 million monthly users in 90 countries and is known for its group chats and disappearing messages. The security researcher who found the flaw was paid a reward of 15,000 dollars under the company's bug-bounty programme. Chatly said it will publish a full technical explanation in two weeks.\""
 },
 {
  "id": "summ-008",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Summarise the following text in no more than 35 words:\n\nThe Pembrook Orchestra will perform all nine Beethoven symphonies over three evenings in July, conducted by Ana Ribeiro, who joined as music director last year. Tickets start at 18 pounds and a limited number of free seats will be given to students. The concerts will be held in the open-air amphitheatre, and if it rains they will move indoors to the town hall, which holds only 400 people. Ribeiro is the first woman to lead the orchestra in its 90-year history. The amphitheatre seats 1,500 and was renovated last year. Each evening will include three symphonies with a short interval, and a pre-concert talk about Beethoven's life will be given by a local professor an hour before each performance."
 },
 {
  "id": "summ-009",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text as exactly 3 bullet points (each line starting with '- '):\n\nEngineers at the Marlow Institute have built a robot that sorts household waste using cameras and a trained neural network. In a pilot at a recycling plant, it identified plastics, metals and paper with 94 percent accuracy and processed 60 items per minute, compared with about 35 for a human worker. The machine struggles with dirty or crushed items, where accuracy falls to 71 percent. Each unit costs 180,000 euros, so the plant expects to recoup its investment only after about five years. The institute, based in Bavaria, has 300 researchers and works with industry on automation. The robot uses a vacuum gripper and a conveyor belt and was trained on 2 million labelled images of waste. The pilot ran for three months at a plant that processes 30,000 tonnes of recyclables a year."
 },
 {
  "id": "summ-010",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text as exactly 3 bullet points (each line starting with '- '):\n\nThe Treaty of Tordesillas, signed in 1494, divided newly discovered lands outside Europe between Spain and Portugal along a line 370 leagues west of the Cape Verde islands. Lands east of the line went to Portugal and those to the west to Spain. The pope had earlier issued a decree that favoured Spain, and Portugal, fearing the loss of its route around Africa, negotiated the shift of the line. The treaty later explained why Portuguese became the language of Brazil, which lay east of the line, while most of the rest of Latin America spoke Spanish. The treaty was signed in a small town in Spain, now in the province of Valladolid. Neither country consulted the peoples who already lived in the lands being divided, and other European powers such as France and England later rejected the agreement. The line was adjusted again in 1529 on the opposite side of the world."
 },
 {
  "id": "summ-011",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Condense this passage in no more than 25 words. Keep only facts stated in the text.\n\n\"A 400-year-old oak tree in the Weald Forest was cut down illegally last week, and police are investigating. The tree, known locally as the Old Warden, was thought to have sheltered soldiers in the English Civil War and was a popular walking destination. The forest trust has offered a 10,000 pound reward for information and says the trunk, about 3 metres around, was removed with a chainsaw at night. The trust will plant a sapling grown from one of the tree's acorns on the same spot. The Weald Forest covers 1,400 hectares and receives about 300,000 visitors a year. Rangers found the stump and sawdust on Monday morning and think the work took several hours. It is a criminal offence to fell a protected tree, punishable with a fine of up to 20,000 pounds, and local woodworkers have been told to report any large oak timber offered for sale.\""
 },
 {
  "id": "summ-012",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text in exactly two sentences:\n\nFarmers in the Valdoro region have begun using drones to spray vineyards, replacing tractors that compact the soil on steep slopes. Each drone covers 20 hectares in a day, about four times the speed of a worker on foot, and cuts pesticide use by 30 percent because it targets only affected vines. The drones cost 25,000 euros each, and a regional subsidy covers half. Union leaders worry that the technology will remove seasonal jobs, while growers say they are short of workers. Valdoro is a wine region of about 12,000 hectares known for its steep terraced vineyards. The drones fly at a height of three metres and are guided by maps made from aerial photographs. Regulations require a licensed pilot for each flight, and a training course for growers starts next month."
 },
 {
  "id": "summ-013",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "A report on remote work surveyed 5,000 employees at 120 companies. Workers who spent three days a week at home reported 12 percent higher job satisfaction than those who were fully in the office, and their managers rated their productivity as unchanged. However, junior staff who worked remotely more than four days a week were 20 percent less likely to be promoted within two years, which the authors attribute to fewer chances to be noticed and mentored. The report recommends hybrid schedules with scheduled in-person days for new hires. The survey was run by an independent research firm in 2023 and covered workers in finance, technology and healthcare administration. Average commuting time saved was 54 minutes a day. The authors noted that results may differ in jobs where workers must be present, and that only 8 percent of respondents were in management.\n\nTL;DR as exactly 3 bullet points (each line starting with '- '), please."
 },
 {
  "id": "summ-014",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Summarise the following text in exactly one sentence:\n\nMarathon runner Daniel Okoye won the Eastport Marathon on Sunday in 2 hours 9 minutes, shaving three minutes off the course record that had stood since 2016. Temperatures stayed near 12 degrees, which organisers called ideal. Some 14,000 runners took part, raising more than 900,000 dollars for children's hospitals. Okoye said he had trained for six months at altitude in Kenya. The previous record of 2:12 was held by Henrik Dahl. The course runs along the coast and finishes in the harbour square, where crowds of about 60,000 gathered. Women's winner Lea Marchetti finished in 2:26, and organisers said the 20th edition would be held on the same weekend next year."
 },
 {
  "id": "summ-017",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Summarise the following text in no more than 35 words:\n\nThe Lakeshore ferry service will be suspended for ten days in September while the pier is rebuilt. The pier has been unsafe since a crack was found in a supporting beam in March, and engineers have limited the ferry to 60 passengers per trip since then. A replacement bus will run between the two towns every 30 minutes, adding about 40 minutes to the journey. The 3.5 million pound project is paid for by the regional transport fund. The ferry carries about 2,000 passengers on a typical summer day and is popular with cyclists and hikers. Passengers with season tickets will be able to use the replacement bus at no extra charge. The pier was built in 1936, and heritage groups want the original wooden railings to be kept."
 },
 {
  "id": "summ-018",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Condense this passage in exactly two sentences. Keep only facts stated in the text.\n\n\"The town council of Ashby has decided to plant 5,000 trees along the main roads over the next three years, aiming to cut summer temperatures in the centre and improve air quality. The 800,000 pound scheme will include volunteers from local schools, who will plant the first 500 saplings this autumn. Critics argue the money should go to repairing potholes, but the council says the trees will also lower flooding risk by absorbing rainwater. The species chosen include lime, hornbeam and field maple, which tolerate pollution and drought. Each tree will cost about 160 pounds to plant and maintain for five years. The council's survey found that 68 percent of residents support the idea, while a small group of shopkeepers worry about leaves blocking drains.\""
 },
 {
  "id": "summ-019",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Condense this passage in exactly one sentence. Keep only facts stated in the text.\n\n\"The Silk Road was not a single road but a network of trade routes linking China with the Mediterranean from about the second century BCE. Merchants rarely travelled the whole distance; goods such as silk, spices and paper passed through many hands. The network also carried ideas and religions, including Buddhism, and diseases such as the plague. It declined as sea routes became cheaper and as the Ottoman Empire controlled key territories in the fifteenth century. Samarkand and Kashgar were among the best-known stopping points, where caravans rested and traded. Camels and horses were the main means of transport across the deserts and mountains. The name 'Silk Road' was only coined in 1877 by a German geographer, long after the routes had fallen into decline.\""
 },
 {
  "id": "summ-024",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Condense this passage in exactly one sentence. Keep only facts stated in the text.\n\n\"The company Nuvo Foods has recalled 120,000 jars of peanut sauce after tests found traces of an allergen that was not listed on the label. The affected jars carry batch numbers beginning with 24B and were sold in 11 countries between January and April. No illnesses have been confirmed. Customers are asked to stop using the product and can return it to the store for a full refund. Nuvo Foods, based in Rotterdam, sells its sauces in more than 30 countries and employs 2,800 people. The company discovered the problem during routine checks after a customer in Germany reported a mild reaction, which was later found to be unrelated. A hotline and a web page with photos of the batch codes have been set up.\""
 },
 {
  "id": "summ-025",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Condense this passage in exactly two sentences. Keep only facts stated in the text.\n\n\"The Kestrel space telescope, launched in 2019, has discovered 54 planets orbiting small red stars, including four that are roughly Earth-sized and lie in the zone where liquid water could exist. Its mission was extended by three years after fuel use proved lower than predicted. Scientists caution that red stars often emit strong flares that could strip atmospheres from nearby planets, so being in the habitable zone does not mean a planet is hospitable. A follow-up telescope is planned for 2031. The telescope is named after a small bird of prey known for hovering motionless while hunting, which reflects how the instrument stares at one patch of sky. It orbits the Sun rather than the Earth, 1.5 million kilometres away. Data are shared publicly after six months, and amateurs have discovered two of the planets.\""
 },
 {
  "id": "summ-026",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "A local charity, Warm Winter, collected 8,400 coats during its autumn drive, double the number from last year. Volunteers cleaned and sorted the coats and will give them out at six community centres from 1 November. The charity says demand has risen because more families are struggling with heating costs. It still needs volunteers on Saturdays and has set up a hotline for people who cannot reach the centres. Last year the drive collected 4,200 coats and had to turn people away at two centres because supplies ran out. This year a clothing retailer donated 1,500 new coats, and a transport company is delivering them for free. Volunteers meet at the warehouse at 9 am, and training is provided on the first day.\n\nTL;DR in no more than 35 words, please."
 },
 {
  "id": "summ-027",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Here's an article my manager sent me. Can you summarise it as exactly 3 bullet points (each line starting with '- ')? I only have a minute.\n\nThe city's new recycling scheme replaces weekly mixed collections with separate bins for paper, glass and food waste, collected on alternating weeks. A pilot in two districts raised recycling rates from 31 to 52 percent in a year and cut the amount of waste sent to landfill by 4,000 tonnes. Some residents complained about the number of bins and about smells in summer, and the council has promised larger food-waste caddies with sealed lids. The scheme will cover all 90,000 homes by 2026. The council collects waste from about 90,000 homes and spends 28 million pounds a year on waste services. Glass is collected in a green box, paper in a blue one and food waste in a brown caddy. Households that put the wrong waste in a bin will first receive a warning leaflet rather than a fine."
 },
 {
  "id": "summ-028",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Here's an article my manager sent me. Can you summarise it in exactly one sentence? I only have a minute.\n\nA new study of 2,400 office workers found that those who took a 15-minute walk after lunch reported 20 percent fewer afternoon energy slumps than those who stayed at their desks. The researchers, based at a university in Leeds, followed participants for eight weeks and used daily questionnaires rather than clinical measurements. They caution that the effect may partly reflect better mood from leaving the office, and they plan a follow-up trial with wearable heart monitors next year. Participants were recruited from 40 firms and asked to rate their alertness on a five-point scale at 3 pm each day. The walking group averaged 3.9 and the desk group 3.2 at the start. The study was paid for by a health insurer, which had no say in the analysis, according to the authors."
 },
 {
  "id": "summ-030",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Condense this passage in no more than 35 words. Keep only facts stated in the text.\n\n\"An international team has mapped the genome of the Atlantic cod, hoping to understand why some populations survive overfishing better than others. They sequenced 1,100 fish from 14 locations and found a cluster of genes linked to growth rate and tolerance of cold water. The results could help managers set fishing limits that protect vulnerable populations, although the authors say more work is needed before the findings guide policy. Atlantic cod were once so abundant that early explorers described rivers thick with fish, but stocks in the western Atlantic collapsed in the early 1990s and some have yet to recover. The study was funded by research councils in four countries, and the genetic data has been made freely available online.\""
 },
 {
  "id": "summ-031",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text in exactly two sentences:\n\nA fire at the Brantley warehouse on Tuesday night destroyed about 2 million dollars worth of electronics. Firefighters from five stations took six hours to bring the blaze under control, and 18 nearby homes were evacuated as a precaution but residents returned the next morning. Investigators say the fire started near a charging area for forklift batteries, although the exact cause is still unknown. No one was injured, and the company says it will reopen a smaller site within a month. The warehouse, built in 2004, stood in an industrial park on the edge of town and stored televisions, laptops and phone accessories for several retailers. Smoke could be seen from 20 kilometres away, and the main road was closed for most of the night. Insurance assessors will visit the site later this week."
 },
 {
  "id": "summ-032",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Summarise the following text in no more than 35 words:\n\nA heatwave in southern Spain pushed temperatures above 44 degrees on Thursday, a record for the month of June. Authorities issued red alerts in six provinces and asked people to avoid outdoor work between noon and 6 pm. Hospitals reported a 25 percent rise in heat-related admissions. The weather agency said the heat should ease at the weekend, when a cooler wind from the Atlantic is expected. The previous June record was 43.1 degrees, set in 2017. Several regions have banned barbecues in forests, and some outdoor events in Seville and Cordoba were postponed. Farmers say that the heat has damaged olive blossoms, and the agriculture ministry will assess the losses next week."
 },
 {
  "id": "summ-034",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Condense this passage as exactly 3 bullet points (each line starting with '- '). Keep only facts stated in the text.\n\n\"A trial of a new migraine drug involved 1,800 patients across 14 countries. Participants who took the drug monthly had on average 4.2 fewer migraine days per month, compared with 1.9 fewer days for those on a placebo. The most common side effects were mild nausea and fatigue, reported by 7 percent of patients. Researchers stressed that the study lasted only six months and excluded people over 65, so long-term safety and effects in older adults remain unknown. The manufacturer will apply for approval next year. Migraine affects about one in seven people worldwide and is more common in women. The drug works by blocking a protein called CGRP, which is involved in pain signalling. The trial was funded by the manufacturer, but the analysis was conducted by an independent university group and published in a medical journal.\""
 },
 {
  "id": "summ-035",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text as exactly 3 bullet points (each line starting with '- '):\n\nUnder the new housing law passed on Friday, landlords in the capital may raise rents by no more than 3 percent a year, down from the 7 percent that was previously allowed. The law also requires landlords to give tenants six months' notice before ending a contract without cause, up from two months. Landlord groups say the cap will push owners to sell, reducing supply, while tenant organisations say it protects families from sudden displacement. The measure takes effect in January and will be reviewed after two years. The capital has about 900,000 rental homes, and rents have risen 35 percent over the past decade. The law passed with 82 votes to 61 after a debate lasting nine hours. The housing minister said that a fund of 50 million euros would support landlords who build new affordable homes."
 },
 {
  "id": "summ-036",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Summarise the following text in exactly one sentence:\n\nCoastal Airways will start a direct flight between Lisbon and Cape Town in October, the first non-stop service between the two cities. The flights will run three times a week on a 220-seat aircraft and take about 11 hours. Introductory fares begin at 389 euros one way. The airline says the route responds to growing demand from tourists and from Portuguese families with ties to southern Africa. Currently passengers between the two cities must change planes in Johannesburg or Dubai, which can add five hours to the trip. Coastal Airways, founded in 2011, flies to 24 destinations and operates a fleet of 18 aircraft. Bookings open on 1 July, and the airline will offer extra baggage on the first week of flights."
 },
 {
  "id": "summ-038",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "The government has announced a plan to give 1.2 million households a one-off payment of 150 pounds towards winter heating bills. The payment will be made automatically in November to people receiving certain benefits, and the 180 million pound cost will come from existing budgets. Charities welcomed the move but said it is smaller than last year's help and does not reach many pensioners on low incomes. Last year's payment was 300 pounds and reached 2.4 million households, which is why charities say this year's scheme is less generous. Energy prices have fallen 22 percent since the peak, but remain well above levels from before 2021. The finance minister said the government would keep the policy under review.\n\nTL;DR in no more than 35 words, please."
 },
 {
  "id": "summ-039",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "When the Aral Sea began to shrink in the 1960s, the Soviet Union was diverting the rivers that fed it to irrigate cotton fields in Uzbekistan and Kazakhstan. By 2007 the sea had lost about 90 percent of its volume, leaving ships stranded on dry sand, ending a fishing industry that once employed 40,000 people, and exposing toxic dust that harmed nearby residents. A dam completed in 2005 helped the northern part recover, with fish catches rising again, but the southern part is expected to remain largely dry. The Aral Sea was once the fourth-largest lake in the world. The rivers that fed it, the Amu Darya and the Syr Darya, still supply irrigation for cotton, which is a very thirsty crop. Governments in the region have planted salt-tolerant shrubs on the exposed seabed to reduce dust storms.\n\nTL;DR as exactly 3 bullet points (each line starting with '- '), please."
 },
 {
  "id": "summ-040",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text in no more than 25 words:\n\nThe Ferris Valley railway, closed in 1966, will partly reopen next year after a campaign by local residents raised 2.5 million pounds. Only 9 of the original 24 kilometres will carry trains at first, run by volunteers using restored steam locomotives at weekends. A further 4 million pounds is needed to rebuild a collapsed bridge before the full line can reopen. Organisers expect the heritage line to bring 60,000 tourists a year to the valley. The Ferris Valley line was closed in 1966 as part of national cuts to rural railways, and its tracks lay overgrown for 30 years before volunteers began clearing them. The reopened stretch will have two stations and a small museum. The campaign group, founded in 2008, has about 1,200 members and receives no government funding."
 },
 {
  "id": "summ-041",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Summarise the following text in exactly two sentences:\n\nThe first successful kidney transplant took place in Boston in 1954, between identical twins, which avoided rejection because the donor and recipient were genetically the same. Surgeon Joseph Murray received the Nobel Prize in 1990 for the work. The breakthrough for unrelated donors came in the 1980s with the drug cyclosporine, which suppresses the immune system more precisely than earlier medicines. Today more than 90,000 kidney transplants are performed worldwide each year, though the waiting list is far longer. The surgeon's patient, Richard Herrick, survived for eight more years with the transplanted kidney, and the donor, his twin Ronald, lived a long life too. Dialysis, which cleans the blood with a machine, was already available in the 1950s but is a burden for patients. Many countries now run organ donation registers."
 },
 {
  "id": "summ-045",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "Here's an article my manager sent me. Can you summarise it in exactly one sentence? I only have a minute.\n\nScientists at an Australian observatory have detected a faint radio signal from a galaxy that is 8 billion light-years away, one of the most distant of its kind ever observed. The signal was caught using a network of 36 dishes working together and took the team nearly three years to confirm. The galaxy is forming stars about 200 times faster than the Milky Way, which suggests that star formation was far more intense in the early universe. The signal is produced by cold gas that is cooling and collapsing into new stars, and it left the galaxy when the universe was only about 5.5 billion years old. The observatory is located in a remote desert, where radio interference is very low. Results are published this week in a leading astronomy journal."
 },
 {
  "id": "summ-047",
  "category": "summarisation",
  "difficulty": "hard",
  "prompt": "Here's an article my manager sent me. Can you summarise it in no more than 25 words? I only have a minute.\n\nCandidates for next month's election in Norvia debated on live television for the first time on Wednesday. Incumbent president Sara Voss defended her record of cutting unemployment from 8 to 5 percent, while challenger Tomas Rhee argued that wages have not kept up with prices and promised to raise the minimum wage by 12 percent. A poll taken after the debate gave Voss 46 percent and Rhee 44 percent, within the margin of error of 3 points, with 10 percent undecided. Norvia has 11 million voters and has been governed by a centre-left coalition since 2019. The debate lasted 90 minutes and was moderated by two journalists. Two other candidates, from the Green party and a small conservative movement, were not invited because their polling is below 5 percent."
 },
 {
  "id": "summ-050",
  "category": "summarisation",
  "difficulty": "easy",
  "prompt": "In 1928 Alexander Fleming noticed that a mould had killed bacteria on one of his culture dishes at St Mary's Hospital in London. He identified the mould as Penicillium and named the active substance penicillin, but he struggled to purify it. It was not until 1940 that Howard Florey and Ernst Chain, working at Oxford, turned it into a usable drug. Mass production during the Second World War saved thousands of soldiers from infected wounds. Fleming was a Scottish bacteriologist who had earlier discovered lysozyme, an enzyme in tears and saliva. He shared the 1945 Nobel Prize with Florey and Chain. Early penicillin was so scarce that doctors sometimes recovered it from patients' urine so that it could be used again.\n\nTL;DR in exactly one sentence, please."
 },
 {
  "id": "ner-001",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Priya Nair works at Google."
 },
 {
  "id": "ner-002",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Amara Singh works at Reliance Industries.\""
 },
 {
  "id": "ner-003",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Tata Consultancy Services hired Anand Rajan in Bengaluru on 1 June 2020.\""
 },
 {
  "id": "ner-004",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Here's a line from a news article:\n\nAmazon expanded its warehouse network along the Amazon River in 2021.\n\nPull out every person, organisation, location and date it mentions, with the type of each."
 },
 {
  "id": "ner-006",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Here's a line from a news article:\n\nSiemens Energy and BP agreed terms with Kenya's government in Nairobi on 2020-11-04.\n\nPull out every person, organisation, location and date it mentions, with the type of each."
 },
 {
  "id": "ner-007",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Jordan Peterson met Jordan's king in Amman."
 },
 {
  "id": "ner-008",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Extract all named entities with their types (PERSON, ORG, LOCATION, DATE) from: 'Dr. Kavita Rao of AIIMS Delhi presented the findings in Geneva on 12 May 2021.'"
 },
 {
  "id": "ner-009",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Extract all named entities with their types (PERSON, ORG, LOCATION, DATE) from: 'John Okafor joined Tata Steel in 9 November 2017.'"
 },
 {
  "id": "ner-010",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Priya Nair and Neha Sharma flew from Sydney to Kerala for a Deutsche Bank meeting in July 2022.\""
 },
 {
  "id": "ner-012",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Here's a line from a news article:\n\nThe Delhi Capitals beat the Mumbai Indians in Dubai on 5 April 2022.\n\nPull out every person, organisation, location and date it mentions, with the type of each."
 },
 {
  "id": "ner-015",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Priya Nair works at Microsoft.\""
 },
 {
  "id": "ner-016",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"The World Health Organization, headquartered in Geneva, praised Kerala's response in 2021.\""
 },
 {
  "id": "ner-017",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Elena Petrova was born in Dubai."
 },
 {
  "id": "ner-018",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Deutsche Bank's chief executive Amara Singh told reporters in Nairobi that the plant would open in 14 Feb 2019.\""
 },
 {
  "id": "ner-020",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Washington crossed the Delaware, according to historian Eric Foner of Columbia University.\""
 },
 {
  "id": "ner-021",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Chen Wei was born in Lagos.\""
 },
 {
  "id": "ner-028",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Here's a line from a news article:\n\nElena Petrova was born in Lisbon.\n\nPull out every person, organisation, location and date it mentions, with the type of each."
 },
 {
  "id": "ner-030",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"After Rahul Mehta's talk in Germany, Tata Steel and Bank of America signed a deal that was announced on December 12, 2019.\""
 },
 {
  "id": "ner-031",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Deutsche Bank announced its quarterly results on 14 Feb 2019."
 },
 {
  "id": "ner-032",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Siemens announced its quarterly results on 9 November 2017.\""
 },
 {
  "id": "ner-033",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Here's a line from a news article:\n\nBetween Nairobi, Pune and Kerala, Omar Haddad visited several Deutsche Bank offices before 9 November 2017.\n\nPull out every person, organisation, location and date it mentions, with the type of each."
 },
 {
  "id": "ner-035",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"David Cohen was born in Cairo.\""
 },
 {
  "id": "ner-037",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Neha Sharma and Rahul Mehta flew from Dubai to Brazil for a Microsoft meeting in 21st June 2023."
 },
 {
  "id": "ner-039",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Vikram Joshi works at Tata Steel.\""
 },
 {
  "id": "ner-041",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "Tag the people, organisations, places and dates in this sentence: Flipkart's chief executive Rahul Mehta told reporters in Tokyo that the plant would open in 1998."
 },
 {
  "id": "ner-043",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Unilever opened a new office in Berlin.\""
 },
 {
  "id": "ner-044",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Neha Sharma works at Flipkart.\""
 },
 {
  "id": "ner-048",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "Tag the people, organisations, places and dates in this sentence: The conference in Tokyo begins on 2024-01-30."
 },
 {
  "id": "ner-049",
  "category": "ner",
  "difficulty": "hard",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Apple's iPhone sales in India rose after Tim Cook visited Mumbai in April 2023.\""
 },
 {
  "id": "ner-050",
  "category": "ner",
  "difficulty": "easy",
  "prompt": "who/what/where/when is mentioned here? list the entities and their types: \"Lars Eriksson joined Reliance Industries in July 2022.\""
 },
 {
  "id": "code_debug-001",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should rotate a list left by k positions (k may exceed the list length) and return the new list; an empty list returns an empty list.\n\n```python\ndef rotate_left(xs, k):\n    if not xs:\n        return []\n    return xs[k:] + xs[:k]\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-003",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "Fix the bug in the following Python function.\nIntended behaviour: it should convert degrees Celsius to degrees Fahrenheit.\n\n```python\ndef c_to_f(c):\n    return c * 5 / 9 + 32\n```"
 },
 {
  "id": "code_debug-004",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "This function should capitalise the first letter of each word and lowercase the rest, words separated by single spaces, but it has a bug. Find and fix it:\n\n```python\ndef capitalize_words(s):\n    return \" \".join(w.upper() for w in s.split())\n```"
 },
 {
  "id": "code_debug-005",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "my `pairs_with_sum` function is giving wrong results and i can't see why. it's supposed to count the pairs of elements at different positions i < j whose sum equals target. here's the code:\n\n```python\ndef pairs_with_sum(xs, target):\n    count = 0\n    for i in range(len(xs)):\n        for j in range(i, len(xs)):\n            if xs[i] + xs[j] == target:\n                count += 1\n    return count\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-006",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should return the length of the longest streak of equal consecutive elements (0 for an empty list), but it has a bug. Find and fix it:\n\n```python\ndef longest_run(xs):\n    best = cur = 0\n    prev = object()\n    for x in xs:\n        if x == prev:\n            cur += 1\n        else:\n            cur = 0\n            prev = x\n        best = max(best, cur)\n    return best\n```"
 },
 {
  "id": "code_debug-008",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return the sum of both diagonals of a square matrix, counting the centre element only once for odd sizes.\n\n```python\ndef diagonal_sum(m):\n    n = len(m)\n    total = sum(m[i][i] + m[i][n - 1 - i] for i in range(n))\n    return total\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-012",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return the largest of three numbers.\n\n```python\ndef max_of_three(a, b, c):\n    if a >= b and a >= c:\n        return a\n    if b >= a:\n        return b\n    return c\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-013",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return True if every (, [ and { in a string is closed by the matching bracket in the right order.\n\n```python\ndef is_balanced(s):\n    pairs = {\")\": \"(\", \"]\": \"[\", \"}\": \"{\"}\n    stack = []\n    for ch in s:\n        if ch in \"([{\":\n            stack.append(ch)\n        elif ch in pairs:\n            if not stack or stack.pop() != pairs[ch]:\n                return False\n    return True\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-014",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return the string reversed, building the result with a loop.\n\n```python\ndef reverse_string(s):\n    result = \"\"\n    for ch in s:\n        result = result + ch\n    return result\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-015",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should return the k most frequent values; ties are broken by the smaller value first, but it has a bug. Find and fix it:\n\n```python\ndef top_k_frequent(xs, k):\n    counts = {}\n    for x in xs:\n        counts[x] = counts.get(x, 0) + 1\n    return sorted(counts, key=lambda x: -counts[x])[:k]\n```"
 },
 {
  "id": "code_debug-017",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "my `dedupe_ignore_case` function is giving wrong results and i can't see why. it's supposed to remove words that repeat an earlier word ignoring case, keeping the first occurrence with its original casing. here's the code:\n\n```python\ndef dedupe_ignore_case(words):\n    seen = set()\n    out = []\n    for w in words:\n        if w not in seen:\n            seen.add(w.lower())\n            out.append(w)\n    return out\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-021",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should count how many times target appears in a list.\n\n```python\ndef count_occurrences(xs, target):\n    count = 0\n    for x in xs:\n        if x != target:\n            count += 1\n    return count\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-022",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "This function should compute base ** exp by repeated multiplication, for integer exp >= 0, but it has a bug. Find and fix it:\n\n```python\ndef power(base, exp):\n    result = 0\n    for _ in range(exp):\n        result *= base\n    return result\n```"
 },
 {
  "id": "code_debug-023",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "my `transpose` function is giving wrong results and i can't see why. it's supposed to transpose a non-empty rectangular matrix given as a list of rows (rows and columns may differ in count). here's the code:\n\n```python\ndef transpose(matrix):\n    return [[row[j] for row in matrix] for j in range(len(matrix))]\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-024",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should run-length encode a string: \"aaabcc\" -> \"a3b1c2\" (empty string -> empty string), but it has a bug. Find and fix it:\n\n```python\ndef compress(s):\n    if not s:\n        return \"\"\n    out = []\n    count = 1\n    for i in range(1, len(s)):\n        if s[i] == s[i - 1]:\n            count += 1\n        else:\n            out.append(s[i - 1] + str(count))\n            count = 1\n    return \"\".join(out)\n```"
 },
 {
  "id": "code_debug-025",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return True if an integer is even, else False (including negative numbers and zero).\n\n```python\ndef is_even(n):\n    return n % 2 == 1\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-029",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "my `join_with_commas` function is giving wrong results and i can't see why. it's supposed to join a list of strings with a comma and a space between items, with nothing after the last item. here's the code:\n\n```python\ndef join_with_commas(items):\n    out = \"\"\n    for it in items:\n        out += it + \", \"\n    return out\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-031",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should return a list where element i is the maximum of the input's first i+1 elements, but it has a bug. Find and fix it:\n\n```python\ndef running_max(xs):\n    out = []\n    current = 0\n    for x in xs:\n        if x > current:\n            current = x\n        out.append(current)\n    return out\n```"
 },
 {
  "id": "code_debug-032",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "Fix the bug in the following Python function.\nIntended behaviour: it should return the sum of the integers from 1 to n inclusive (0 when n is 0).\n\n```python\ndef sum_to_n(n):\n    total = 0\n    for i in range(1, n):\n        total += i\n    return total\n```"
 },
 {
  "id": "code_debug-034",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should return the binary representation of a non-negative integer as a string without any prefix.\n\n```python\ndef to_binary(n):\n    return bin(n)\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-036",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should count the vowels (a, e, i, o, u, case-insensitive) in a string.\n\n```python\ndef count_vowels(s):\n    return sum(1 for c in s.lower() if c in \"aeio\")\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-037",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "Fix the bug in the following Python function.\nIntended behaviour: it should return the smallest element of a non-empty list of numbers.\n\n```python\ndef min_in_list(xs):\n    m = 0\n    for x in xs:\n        if x < m:\n            m = x\n    return m\n```"
 },
 {
  "id": "code_debug-038",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "my `kth_largest_distinct` function is giving wrong results and i can't see why. it's supposed to return the k-th largest DISTINCT value of a list (k starts at 1), or None if there are fewer than k distinct values. here's the code:\n\n```python\ndef kth_largest_distinct(xs, k):\n    vals = sorted(xs, reverse=True)\n    return vals[k - 1] if 1 <= k <= len(vals) else None\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-039",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "my `is_prime` function is giving wrong results and i can't see why. it's supposed to return True if n is a prime number, else False (n may be 0, 1 or negative). here's the code:\n\n```python\ndef is_prime(n):\n    if n < 2:\n        return False\n    for d in range(2, int(n ** 0.5)):\n        if n % d == 0:\n            return False\n    return True\n```\nwhat's wrong with it?"
 },
 {
  "id": "code_debug-040",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "This function should return the last element of a list, or None if the list is empty, but it has a bug. Find and fix it:\n\n```python\ndef last_element(xs):\n    return xs[len(xs)] if xs else None\n```"
 },
 {
  "id": "code_debug-041",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "I'm reviewing a teammate's pull request and a QA test is failing on this helper. The docstring says it should group words by length: return a dict mapping each length to the list of words with that length, in input order.\n\n```python\ndef group_by_length(words):\n    groups = {}\n    for w in words:\n        groups[len(w)] = [w]\n    return groups\n```\n\nCan you work out the problem and give me the corrected function?"
 },
 {
  "id": "code_debug-045",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "This function should return True if a year is a leap year in the Gregorian calendar, but it has a bug. Find and fix it:\n\n```python\ndef is_leap_year(y):\n    return (y % 4 == 0 and y % 100 != 0) or y % 100 == 0\n```"
 },
 {
  "id": "code_debug-046",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should return a NEW sorted list with value inserted into an already-sorted list; do not modify the input list, but it has a bug. Find and fix it:\n\n```python\ndef insert_sorted(xs, v):\n    out = xs\n    i = 0\n    while i < len(out) and out[i] < v:\n        i += 1\n    out.insert(i, v)\n    return out\n```"
 },
 {
  "id": "code_debug-047",
  "category": "code_debug",
  "difficulty": "hard",
  "prompt": "This function should return a new list with every occurrence of val removed (the input list should be left unchanged), but it has a bug. Find and fix it:\n\n```python\ndef remove_all(lst, val):\n    for x in lst:\n        if x == val:\n            lst.remove(x)\n    return lst\n```"
 },
 {
  "id": "code_debug-050",
  "category": "code_debug",
  "difficulty": "easy",
  "prompt": "Fix the bug in the following Python function.\nIntended behaviour: it should restrict x to the range [lo, hi].\n\n```python\ndef clamp(x, lo, hi):\n    return min(lo, max(x, hi))\n```"
 },
 {
  "id": "logic-002",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three runners - Gus, Kofi and Jia - finished a race with no ties. Kofi did not finish first. Kofi did not finish second. Jia did not finish first.\n\nWho finished first?"
 },
 {
  "id": "logic-003",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Four runners - Fay, Omar, Pia and Uma - finished a race with no ties. Fay did not finish fourth. Omar finished ahead of Fay. Uma finished third.\n\nWho finished fourth?"
 },
 {
  "id": "logic-004",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five runners - Hana, Dia, Ben, Fay and Lena - finished a race with no ties.\n\n1. Hana did not finish second.\n2. Dia finished somewhere between Hana and Ben.\n3. Lena did not finish third.\n4. Lena did not finish second.\n5. Fay beat Ben.\n6. Ben did not finish fourth.\n7. Dia did not finish third.\n8. Fay did not finish second.\n\nWho finished first?"
 },
 {
  "id": "logic-006",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five runners - Cy, Raj, Kofi, Fay and Pia - finished a race with no ties.\n\n- Kofi finished somewhere between Cy and Pia.\n- Raj beat Fay.\n- Raj did not finish third.\n- Pia finished immediately after Fay.\n- Kofi finished ahead of Pia.\n- Raj did not finish second.\n\nWho finished fourth?"
 },
 {
  "id": "logic-009",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five runners - Kofi, Ben, Ana, Raj and Pia - finished a race with no ties.\n\n- Exactly one runner finished between Ana and Raj.\n- Kofi finished ahead of Raj.\n- Raj did not finish fourth.\n- Kofi did not finish first.\n- Raj finished ahead of Pia.\n- Ana did not finish first.\n\nWho finished second?"
 },
 {
  "id": "logic-010",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five friends - Maya, Raj, Jia, Sara and Pia - each own a different pet: cat, dog, rabbit, fish, parrot. Sara does not own the parrot. The rabbit belongs to either Maya or Pia. The fish belongs to either Jia or Pia. The cat belongs to either Sara or Pia. Raj does not own the parrot. The fish belongs to either Jia or Sara. The rabbit belongs to either Maya or Raj.\n\nWho owns the rabbit?"
 },
 {
  "id": "logic-011",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Four runners - Raj, Fay, Lena and Cy - finished a race with no ties. Lena finished third. Cy did not finish fourth. Fay finished second.\n\nWho finished first?"
 },
 {
  "id": "logic-012",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Four friends - Ivan, Eli, Ben and Kofi - each own a different pet: dog, rabbit, cat, fish.\n\n- Kofi is not the owner of the fish.\n- The fish belongs to either Ben or Kofi.\n- Kofi does not own the cat.\n- Eli owns the rabbit.\n\nWho owns the cat?"
 },
 {
  "id": "logic-014",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five runners - Lena, Omar, Hana, Cy and Noor - finished a race with no ties.\n\n- Cy did not finish fourth.\n- Lena finished ahead of Cy.\n- Lena beat Noor.\n- Noor did not finish fourth.\n- Hana beat Lena.\n- Omar finished somewhere between Lena and Cy.\n\nWho finished first?"
 },
 {
  "id": "logic-015",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five people - Pia, Cy, Uma, Fay and Ana - stand in a single queue at a ticket counter.\n\n- Uma is in the queue between Pia and Ana.\n- Fay is in the queue between Pia and Ana.\n- Pia is behind Fay in the queue.\n- Ana is not second in line.\n- Fay is behind Cy in the queue.\n- Cy is not second in line.\n\nWho is third in line?"
 },
 {
  "id": "logic-016",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five runners - Omar, Raj, Gus, Fay and Cy - finished a race with no ties.\n\n- Omar did not finish fourth.\n- Exactly one runner finished between Raj and Gus.\n- Cy did not finish first.\n- Raj finished somewhere between Gus and Cy.\n- Fay did not finish second.\n- Fay beat Omar.\n\nWho finished fifth?"
 },
 {
  "id": "logic-017",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three runners - Ivan, Uma and Ana - finished a race with no ties. Ivan beat Ana. Ivan beat Uma. Uma did not finish second.\n\nWho finished first?"
 },
 {
  "id": "logic-019",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five people - Kofi, Uma, Noor, Fay and Ben - stand in a single queue at a ticket counter.\n\n- Fay is behind Kofi in the queue.\n- Kofi is in the queue between Uma and Ben.\n- Fay is not fourth in line.\n- Noor is in the queue between Kofi and Fay.\n- Ben is not first in line.\n- Noor is not fourth in line.\n\nWho is third in line?"
 },
 {
  "id": "logic-021",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five friends - Raj, Ivan, Lena, Ben and Sara - each own a different pet: fish, cat, rabbit, parrot, dog. Raj is not the owner of the dog. Raj is not the owner of the rabbit. Ivan does not own the rabbit. The fish belongs to either Ben or Sara. Sara is not the owner of the rabbit. The fish belongs to either Ivan or Ben. The cat belongs to either Lena or Sara.\n\nWho owns the parrot?"
 },
 {
  "id": "logic-024",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three friends - Hana, Gus and Cy - each own a different pet: parrot, fish, rabbit.\n\n1. Gus is not the owner of the parrot.\n2. The rabbit belongs to either Gus or Cy.\n3. Gus does not own the rabbit.\n\nWho owns the parrot?"
 },
 {
  "id": "logic-025",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Pia, Raj, Jia, Eli and Ivan each picked up a different book from a shelf holding a diary, atlas, manual, novel or cookbook.\n\n1. Jia did not pick the atlas.\n2. Pia did not pick the diary.\n3. The manual went to one of Eli and Ivan.\n4. Raj did not pick the diary.\n5. Pia did not pick the novel.\n6. The atlas went to one of Jia and Eli.\n\nWho ended up with the novel?"
 },
 {
  "id": "logic-026",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "At a cafe, four people - Fay, Omar, Noor and Eli - each ordered a different drink: coffee, juice, milk, lassi.\n\n1. Either Fay or Eli ordered the coffee.\n2. Eli ordered the lassi.\n3. Noor did not order the juice.\n\nWho ordered the coffee?"
 },
 {
  "id": "logic-029",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three boxes are labelled \"Apples\", \"Oranges\" and \"Mixed\". Each box really contains one of: only apples, only oranges, or a mix of both, and every label is wrong. You take one fruit out of the box labelled \"Mixed\" and it is an apple. What does the box labelled \"Oranges\" contain? (answer: apples, oranges or mixed)"
 },
 {
  "id": "logic-034",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Three neighbours - Hari, Isha and Jay - each keep a different pet (cat, dog or fish) and each drink a different beverage (milk, tea or soda). Hari keeps the cat. The dog owner drinks tea. Isha drinks soda. Jay does not keep the fish. Who keeps the fish and what do they drink? Answer with the person's name only."
 },
 {
  "id": "logic-037",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Four people - Ben, Gus, Uma and Maya - stand in a single queue at a ticket counter.\n\n- Ben is behind Uma in the queue.\n- Ben and Uma have exactly one person between them.\n- Gus is in the queue between Ben and Maya.\n\nWho is fourth in line?"
 },
 {
  "id": "logic-038",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "At a cafe, three people - Eli, Fay and Gus - each ordered a different drink: lassi, juice, tea. Either Fay or Gus ordered the tea. Either Eli or Fay ordered the lassi. Either Fay or Gus ordered the lassi.\n\nWho ordered the tea?"
 },
 {
  "id": "logic-040",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "At a cafe, five people - Lena, Gus, Tomas, Eli and Omar - each ordered a different drink: coffee, tea, milk, lassi, juice.\n\n1. Omar did not order the lassi.\n2. Tomas did not order the coffee.\n3. Lena did not order the coffee.\n4. Either Gus or Eli ordered the milk.\n5. Lena did not order the tea.\n6. Eli did not order the coffee.\n7. Omar did not order the coffee.\n8. Omar did not order the tea.\n\nWho ordered the lassi?"
 },
 {
  "id": "logic-042",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five friends - Lena, Ben, Noor, Ivan and Maya - each own a different pet: rabbit, fish, parrot, cat, dog.\n\n- The cat belongs to either Ben or Maya.\n- The fish belongs to either Lena or Noor.\n- The parrot belongs to either Ivan or Maya.\n- Ivan is not the owner of the rabbit.\n- Lena is not the owner of the rabbit.\n- The parrot belongs to either Ben or Maya.\n\nWho owns the fish?"
 },
 {
  "id": "logic-044",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three runners - Lena, Fay and Cy - finished a race with no ties.\n\n1. Fay did not finish second.\n2. Lena beat Cy.\n3. Fay did not finish first.\n\nWho finished first?"
 },
 {
  "id": "logic-045",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "At a cafe, three people - Sara, Cy and Ivan - each ordered a different drink: coffee, lassi, milk. Either Sara or Cy ordered the milk. Sara did not order the milk. Either Cy or Ivan ordered the lassi.\n\nWho ordered the lassi?"
 },
 {
  "id": "logic-046",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five friends - Raj, Ben, Eli, Ana and Pia - each own a different pet: cat, fish, rabbit, dog, parrot. The cat belongs to either Raj or Ben. Raj does not own the parrot. Pia is not the owner of the fish. Ana does not own the fish. Pia does not own the rabbit. The parrot belongs to either Raj or Eli. Raj is not the owner of the cat.\n\nWho owns the cat?"
 },
 {
  "id": "logic-047",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "At a cafe, four people - Jia, Uma, Ben and Gus - each ordered a different drink: tea, milk, lassi, juice.\n\n1. Either Jia or Gus ordered the lassi.\n2. Jia ordered the milk.\n3. Ben did not order the tea.\n\nWho ordered the juice?"
 },
 {
  "id": "logic-048",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "Three people - Tomas, Omar and Pia - stand in a single queue at a ticket counter.\n\n- Omar is not second in line.\n- Tomas is not second in line.\n- Omar is behind Pia in the queue.\n\nWho is first in line?"
 },
 {
  "id": "logic-049",
  "category": "logic",
  "difficulty": "easy",
  "prompt": "An office has three staff - Sara, Maya and Dia - and each sits in a different coloured room (white, blue, green).\n\n- The blue room is occupied by Sara or Dia (not by anyone else).\n- The blue room is occupied by Maya or Dia (not by anyone else).\n- Maya is not in the green room.\n\nWhose is the blue room?"
 },
 {
  "id": "logic-050",
  "category": "logic",
  "difficulty": "hard",
  "prompt": "Five friends - Hana, Pia, Eli, Uma and Cy - each own a different pet: cat, parrot, rabbit, fish, dog.\n\n- Cy does not own the parrot.\n- Cy is not the owner of the dog.\n- Uma does not own the parrot.\n- The rabbit belongs to either Hana or Uma.\n- The rabbit belongs to either Hana or Pia.\n- The cat belongs to either Pia or Eli.\n- Pia is not the owner of the parrot.\n\nWho owns the dog?"
 },
 {
  "id": "code_gen-001",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `is_valid_ipv4(s)`; it returns True if s is a valid dotted-decimal IPv4 address: exactly four numbers separated by dots, each 0-255 written without leading zeros (\"0\" is fine, \"01\" is not) and containing only digits. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-002",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "need a python function `remove_vowels(s)` - it returns s with all vowels (a, e, i, o, u in either case) removed. can you write it?"
 },
 {
  "id": "code_gen-003",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "need a python function `multiples(n, k)` - it returns a list of the first k positive multiples of n, in increasing order (empty list if k <= 0). can you write it?"
 },
 {
  "id": "code_gen-004",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "need a python function `to_camel_case(s)` - it converts a snake_case string to camelCase: \"hello_big_world\" -> \"helloBigWorld\" (a string with no underscore is returned unchanged). can you write it?"
 },
 {
  "id": "code_gen-006",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `word_wrap(text, width)`; it greedily wraps the words of text into lines of at most width characters, joining words with single spaces; a word longer than width gets its own line; returns the list of lines (empty list if there are no words). Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-007",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `middle_element(xs)`; it returns the middle element of the list xs; for an even-length list return the first of the two middle elements; return None for an empty list. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-009",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Implement `count_positive(xs)` in Python. It returns how many numbers in xs are strictly greater than zero."
 },
 {
  "id": "code_gen-012",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "need a python function `is_anagram(a, b)` - it returns True if the strings a and b are anagrams of each other, ignoring case and spaces. can you write it?"
 },
 {
  "id": "code_gen-014",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Write a Python function `char_frequency(s)` that returns a dict mapping each character of s to the number of times it appears."
 },
 {
  "id": "code_gen-015",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "need a python function `number_to_words(n)` - it converts an integer 0 <= n <= 999 to English words in lowercase, e.g. 0 -> \"zero\", 21 -> \"twenty-one\", 100 -> \"one hundred\", 342 -> \"three hundred forty-two\" (no word \"and\"; hyphen only between tens and ones words from 21 to 99). can you write it?"
 },
 {
  "id": "code_gen-016",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Implement `reverse_words(s)` in Python. It returns the words of the string s in reverse order, joined by single spaces (extra whitespace in s is ignored)."
 },
 {
  "id": "code_gen-019",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `edit_distance(a, b)`; it returns the Levenshtein edit distance between strings a and b: the minimum number of single-character insertions, deletions and substitutions to turn a into b. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-022",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Implement `longest_word(s)` in Python. It returns the longest whitespace-separated word in s (the first one if there is a tie), or the empty string if s has no words."
 },
 {
  "id": "code_gen-023",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Write a Python function `min_coins(coins, amount)` that returns the minimum number of coins from the list coins (unlimited supply of each denomination) needed to make exactly amount, or -1 if it is impossible; amount 0 needs 0 coins."
 },
 {
  "id": "code_gen-024",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `lcs_length(a, b)`; it returns the length of the longest common SUBSEQUENCE of strings a and b (characters in order but not necessarily contiguous). Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-027",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `longest_unique_substring(s)`; it returns the length of the longest substring of s that contains no repeated character. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-028",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `is_sorted(xs)`; it returns True if the list xs is in non-decreasing order (an empty or one-element list counts as sorted). Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-033",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Write a Python function `longest_common_prefix(strs)` that returns the longest common prefix shared by all strings in the list strs (empty string if there is none or the list is empty)."
 },
 {
  "id": "code_gen-034",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Write a Python function `is_valid_password(p)` that returns True only if the string p has at least 8 characters and contains at least one uppercase letter, one lowercase letter and one digit."
 },
 {
  "id": "code_gen-037",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Implement `list_diff(a, b)` in Python. It returns the elements of list a that do not appear in list b, keeping their order and duplicates from a."
 },
 {
  "id": "code_gen-038",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "need a python function `group_anagrams(words)` - it groups words that are anagrams of each other; returns a list of groups, where each group keeps input order and the groups are ordered by the first appearance of their earliest member. can you write it?"
 },
 {
  "id": "code_gen-041",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Implement `max_subarray_sum(nums)` in Python. It returns the largest sum of a non-empty contiguous subarray of nums (0 if nums is empty); the answer may be negative if all numbers are negative."
 },
 {
  "id": "code_gen-042",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Write a Python function `to_base(n, b)` that converts the integer n (possibly negative or zero) to a string in base b, where 2 <= b <= 16, using digits 0-9 and uppercase A-F, with a leading '-' for negative numbers and no leading zeros."
 },
 {
  "id": "code_gen-043",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `repeat_with_dash(s, n)`; it returns the string s repeated n times, separated by single dashes (\"ab\", 3 -> \"ab-ab-ab\"); returns \"\" when n <= 0. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-044",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Write a Python function `two_sum(nums, target)` that returns [i, j] with i < j and nums[i] + nums[j] == target; if several pairs work, return the one with the smallest i, and for that i the smallest j; return None if no pair exists."
 },
 {
  "id": "code_gen-045",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Write a Python function `swap_pairs(xs)` that returns a new list in which each adjacent pair of elements is swapped, [1,2,3,4,5] -> [2,1,4,3,5]; a final unpaired element stays in place."
 },
 {
  "id": "code_gen-046",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "I'm building a small utility module and I need one more helper. Please write the Python function `pascal_row(n)`; it returns row n (0-indexed) of Pascal's triangle as a list of integers; pascal_row(0) is [1]. Edge cases matter, so please handle them."
 },
 {
  "id": "code_gen-048",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Write a Python function `matrix_multiply(a, b)` that multiplies matrix a (m x n) by matrix b (n x p), both lists of rows, and returns the m x p product as a list of rows."
 },
 {
  "id": "code_gen-049",
  "category": "code_gen",
  "difficulty": "hard",
  "prompt": "Implement `int_to_roman(n)` in Python. It converts an integer 1 <= n <= 3999 to its standard Roman numeral (using subtractive notation such as IV and CM)."
 },
 {
  "id": "code_gen-050",
  "category": "code_gen",
  "difficulty": "easy",
  "prompt": "Implement `product_of_list(xs)` in Python. It returns the product of all numbers in the list xs (1 for an empty list)."
 }
]

In [ ]:
%%writefile cpu_sample_ids.json
{
 "ids": [
  "code_debug-015",
  "code_debug-017",
  "code_debug-024",
  "code_debug-029",
  "code_debug-034",
  "code_gen-007",
  "code_gen-015",
  "code_gen-027",
  "code_gen-037",
  "code_gen-044",
  "fact-009",
  "fact-010",
  "fact-013",
  "fact-023",
  "fact-043",
  "logic-012",
  "logic-017",
  "logic-019",
  "logic-024",
  "logic-050",
  "math-007",
  "math-022",
  "math-031",
  "math-043",
  "math-046",
  "ner-002",
  "ner-004",
  "ner-008",
  "ner-017",
  "ner-032",
  "sent-002",
  "sent-015",
  "sent-034",
  "sent-035",
  "sent-049",
  "summ-013",
  "summ-019",
  "summ-024",
  "summ-025",
  "summ-041"
 ]
}

In [ ]:
import json, collections
t = json.load(open('tasks_dev.json'))
print(len(t), collections.Counter(x['category'] for x in t))
assert len(t) == 240
assert not any('gold' in x for x in t)

## Section 1: core models (resumable)

In [ ]:
!python phase3_run.py --models llama3.2:3b phi3.5:3.8b gemma3:4b --out answers_gpu_b.jsonl --run-id phase3_gpu_b

## Section 2 (OPTIONAL): GPU-server tier, outside the 16 models
Skip this cell if short on session time. Run before zipping.

In [ ]:
!python phase3_run.py --models gemma3:12b phi4:14b qwen2.5:14b --out answers_gpu_b.jsonl --run-id phase3_gpu_server

## Quick look + zip for download

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open('answers_gpu_b.jsonl') if l.strip()]
df = pd.DataFrame(rows)
print('records', len(df), 'errors', df['error'].notna().sum())
print(df.groupby('model')[['tokens_out', 'latency_ms']].median())
print(df.groupby(['model', 'done_reason']).size())

In [ ]:
!zip -q phase3_local_gpu_b.zip answers_gpu_b.jsonl tasks_dev.json system_prompts.json phase3_run.py cpu_sample_ids.json && ls -lh phase3_local_gpu_b.zip